In [ ]:
%load_ext autoreload
%autoreload 2

# Tabel 111 `minema` + genitiivis obliikvid (et leida vigaselt märgendatud kohad)

Sisuliselt on tegemist skript 110 klooniga, kus on teine käänetefilter.

Tasakaalus korpusest kogutakse kokku verbi `minema` esinemused, kus:
* verbi lemma on "minema";
* verbil on (vahetu) alluv `deprel=obl` ja `case=[gen]`: genitiivis obliikva;

Tabelisse salvestatakse järgmised veerud (ilma hiljem lisatavate metaandmete veergudeta):
|veerg|märkus|
|---|----|
|`sent_id`|lause unikaalne id korpuses|
|`text_uid`|teksti unikaalne id|
|`verb`|leitud verbi lemma, antud tabelis alati `minema`|
|`verb_feats`|verbi `feats` väljal olevate tunnuste nimed ühe stringina|
|`obl`|obliikva lemma|
|`obl_pos`|obliikva sõnaliik|
|`obl_case`|obliikva kääne|
|`subj`|verbi vahetu(te) subjekti(de) vorm(id), eraldajaks koma|
|`subj_pos`|subjekti(de) sõnaliigid, eraldajaks koma|
|`text`|fraasi osa: verb + obliikva + võimalik subjekt, lauses esinemise järjekorras|
|`sentence_text`|terve lause tekst|

In [ ]:
import re
from collections import defaultdict
from datetime import datetime

import set_root
import pandas as pd
from config import DATA_DERIVED_QUERY_RESULTS_DIR, METADATA_TSV_DELIMITER, METADATA_TSV_FILE, SOURCE_CONLLU_FILE, TMP_DIR
from data_helpers.meta import enrich_collection_with_metadata
from data_helpers.syntax.conllu_reader import CoNNLUReader
from data_helpers.syntax.render_graphviz import render_syntax_graph

collection_output_file = DATA_DERIVED_QUERY_RESULTS_DIR / "111_notebook_collection.tsv"


In [ ]:
my_reader = CoNNLUReader(SOURCE_CONLLU_FILE)
my_verbs = ["minema"]

CASES = ["gen"]

In [ ]:
%%time
from data_helpers.syntax.constants import SUBJECT_DEPRELS, VERB_POS
from data_helpers.syntax.list_utils import ListUtils
date_time = datetime.now().strftime("%Y%m%d-%H%M%S")

collected_data = []
count = 0
render_occurrence_by_uid = defaultdict(int)
for collection_id, graph in my_reader.get_sentences():
    # matrix for node distances
    dpath = graph.get_distances_matrix()
    
    # verb nodes
    verb_nodes = [v for v in graph.get_nodes_by_attributes(attrname="POS", attrvalue=VERB_POS) if graph.nodes[v]["lemma"] in my_verbs]
    if not len(verb_nodes): 
        continue
    
    # obl nodes
    obl_nodes = graph.get_nodes_by_attributes(attrname="deprel", attrvalue="obl")
    if not len(obl_nodes): 
        continue
    
   
    subj_nodes = graph.get_nodes_by_attributes(attrname="deprel", attrvalue=SUBJECT_DEPRELS)
    
    
    for verb in verb_nodes:
        # childnodes
        kids = [k for k in dpath[verb] if dpath[verb][k] == 1]
        obl_kids = ListUtils.list_intersection(kids, obl_nodes)
        subj_kids = ListUtils.list_intersection(kids, subj_nodes)
        
        for obl in obl_kids:
            obl_case = graph.get_node_case(obl)
            if not obl_case in CASES:
                continue
            
            
            sentence_uid = graph.get_metadata('sent_id') or graph.get_metadata('text_uid') or collection_id
            safe_sentence_uid = re.sub(r"[^0-9A-Za-z._-]+", "_", str(sentence_uid))
            render_occurrence_by_uid[safe_sentence_uid] += 1
            render_syntax_graph(
                graph=graph,
                filename=f"{safe_sentence_uid}_{render_occurrence_by_uid[safe_sentence_uid]:03d}.svg",
                highlight_groups=[
                    {"nodes": [verb], "color": "red"},
                    {"nodes": [obl], "color": "gold"},
                    {"nodes": subj_kids, "color": "palegreen"},
                ],
                output_dir=TMP_DIR,
            )
            d = {
                'sent_id':  graph.get_metadata('sent_id'),
                'text_uid':  graph.get_metadata('text_uid'),
                
                'verb':  graph.nodes[verb]["lemma"],
                'verb_feats':  " ".join(graph.nodes[verb]["feats"].keys()),
                
                'obl':  graph.nodes[obl]["lemma"],
                'obl_pos':  graph.get_node_pos(obl),
                'obl_case':  obl_case,
                
                'subj':  [graph.nodes[s]["form"] for s in subj_kids],
                'subj_pos':  [graph.get_node_pos(s) for s in subj_kids],
                'text': " ".join(
                        [graph.nodes[n]["form"] for n in sorted([verb] + [obl] + subj_kids)]
                ),
                'sentence_text':  graph.get_metadata('text'),
            }
            
            collected_data.append(d)

In [ ]:
collected_data

In [ ]:
df_collected = pd.DataFrame(collected_data)
if not df_collected.empty:
    for column in ["subj", "subj_pos"]:
        if column in df_collected.columns:
            df_collected[column] = df_collected[column].apply(lambda values: ", ".join(values) if isinstance(values, list) else values)

collection_output_file.parent.mkdir(parents=True, exist_ok=True)
df_collected.to_csv(collection_output_file, index=None, sep="\t")
df_collected

### Metaandmete lisamine

Metaandmete failist lisatakse kõik veerud.


In [ ]:
collected_uid_column = "text_uid"
metadata_uid_column = "uid"

metadata_output_file = collection_output_file.with_name(
    f"{collection_output_file.stem}_with_metadata.tsv"
)

df_enriched = enrich_collection_with_metadata(
    df_collected=df_collected,
    metadata_file=METADATA_TSV_FILE,
    collected_uid_column=collected_uid_column,
    metadata_uid_column=metadata_uid_column,
    output_file=metadata_output_file,
    metadata_delimiter=METADATA_TSV_DELIMITER,
    output_delimiter="\t",
)
df_enriched